Importing Libraries

In [31]:
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

Loading playlist data

In [19]:
playlists = pd.read_csv("playlists.csv")
playlists.head()

,track_id,track_name,playlist_id,playlist_name
0,4qEoqyPbLYnLOii6mKlIjI,"Determinate - From ""Lemonade Mouth""",7jQHOrErpLMStcUUSavQWR,Post teen pop
1,5lz0NiPw32Gq4kMIUJvuw2,"Take On the World - Theme Song From ""Girl Meet...",7jQHOrErpLMStcUUSavQWR,Post teen pop
2,1rM0CnyUiiw6A9CHJRXjZA,Chillin' Like a Villain,7jQHOrErpLMStcUUSavQWR,Post teen pop
3,744ZuzjXQmoJmOdk2I1ym9,"Keep It Undercover - Theme Song From ""K.C. Und...",7jQHOrErpLMStcUUSavQWR,Post teen pop
4,6VcKAd94eVxNyudlWN91GH,"Time of Our Lives - Main Title Theme From ""I D...",7jQHOrErpLMStcUUSavQWR,Post teen pop


Data Preparation for evaluation : Grouping the data by Playlist ID and Name

In [ ]:
# Grouping by the playlist id and name
# converting track ids into a list for each playlist
playlist_data = playlists.groupby(['playlist_id', 'playlist_name'])['track_id'].apply(list).reset_index()
playlist_data.head()

,playlist_id,playlist_name,track_id
0,0BwUQpqHSlC2YfKwOp2dQV,Gangsta Rap ð Rap Party,"[5jAeM4WKpqDX9wDJGa7IFq, 3QbglwFgPJDHq7TbmUb3I..."
1,0DdGqNa18DwYyfIR05OrW1,Viral Southern Hip Hop,"[53v6w87pm4hVFwC7GlT2JI, 0M9mH33xccGM4ITZIGwX2..."
2,0QN4FeJQ1mpCygRg9r2JIK,Permanent wave ð,"[6zfczP87XO2SxWlQtnjFNa, 6iRrt1UtrgsxsOZUjp6Vg..."
3,0TT57Pe3RZNGCy98G1UQpM,TROPICALð´,"[3zbSYBDUbDEbqUIlCmOw09, 1siV5UzcXuFK5tMUVQ5Ki..."
4,0VS9w0NY4KXfLORkhY81s8,Permanent Wave Cafe,"[42T2QQv3xgBlpQxaSP7lnK, 3jxrPXWriZPIXKGbuniXh..."


The number of playlists to be used in evaluation.

In [35]:
print ("Number of playlists to be used in evaluation : ",len(playlist_data))

Number of playlists to be used in evaluation :  51


Getting the tracks of each playlist ready for the evaluation

In [40]:
input_tracks_list = [] # a variable to store the lists of input_tracks from each playlist
expected_tracks_list = [] # a variable to store the lists of expected_tracks (the ground truth) from each playlist

for _, row in playlist_data.iterrows():
    # get tracks of each playlist
    tracks = row['track_id'].copy()

    # shuffle the tracks of each playlist
    np.random.shuffle(tracks)

    # split the tracks of each playlist (80%, 20%)
    # 80% for input
    # 20% for expected tracks ground truth
    input_tracks, expected_tracks = train_test_split(tracks, test_size=0.2, random_state=42)
    
    # storing the lists of tracks from each playlist in the list
    input_tracks_list.append(input_tracks)
    expected_tracks_list.append(expected_tracks)

    

In [49]:
print (input_tracks_list[0])
print ("Number of the lists of input_tracks : ", len(input_tracks_list))
print ("Number of the lists of expected_tracks : ", len(expected_tracks_list))

['3YreXzJAZRrchO72v5G5K9', '5sH7vJ3shCpWGpsHMjwkfN', '3PgRUeQuuY94FkKdDBbKvq', '0uYmQ3X53P03KWj83u5I59', '3BYY2756aaf3cCfrqaVRAt', '1fG01hFiFGXZKRMHXsBlyf', '5HEm9RTM8fIuzKa1RBj6xZ', '3N5XOukuZ6JCJHoHVRowS7', '1ISiF8S1Yehu7OrSJU8Kct', '3i2GE43eD2KiN0B5bWNvo2', '5jAeM4WKpqDX9wDJGa7IFq', '4TnjEaWOeW0eKTKIEvJyCa', '0Lfz9oMafqp2UFH8xkZgDY', '1VAFud1nSkqa7efJBxONAz', '5LUNEABcjkQ7jCSpu3uBPq', '0zLXP2PI5JG3pZ2KaqEWS6', '3QbglwFgPJDHq7TbmUb3IE', '6djzhYE1NxgS7YgnZZEqnN', '6AFWx8JqsslV2MsWhpGjnR', '3cyJAZm2r79wdzrsQtv5Ob', '3OueBUZsmQERpu4hSF0jCr', '4mTtMweEZoyyjq7emPX5hQ', '7JXLHs3JxQ0sibWrYeV1co', '2jv5jGgC1iFQ9Ury2ci9Y4']
Number of the lists of input_tracks :  51
Number of the lists of expected_tracks :  51


In [50]:
# Euclidean API URL
modelEuclideanURL = "http://127.0.0.1:8080/api/recommend/"
baselineCosineURL = "http://127.0.0.1:8080/api/recommend/baseline/cosine/"
baselineRandomByArtistURL = "http://127.0.0.1:8080/api/recommend/baseline/random/"

# for each playlist, inputs (list of tracks) are given to recommender logic
for track_lists in input_tracks_list: 
    # data preparation for input (for Cosine and Euclidean)
    json_payload = {
        "track_ids" : track_lists,
        # no preference is also okay (here it is default weight of 1.0)
        "preferences": {
            "energy_weight": 1.0,
            "tempo_weight": 1.0
        }
    }

    # for random by artist model
    json_payload_no_pref = {
        "track_ids" : track_lists
    }
    
    # post with input to url 
    response = requests.post(modelEuclideanURL, json= json_payload)     #Euclidean
    response_cosine = requests.post(baselineCosineURL, json= json_payload)     #Cosine
    response_random = requests.post(baselineRandomByArtistURL, json= json_payload_no_pref)       #Random by artist

    data = response.json()          #Euclidean  
    data_cosine = response_cosine.json()        #Cosine 
    data_random = response_random.json()        #Random by artist
    print(data["track_id"], data["track_id"], data_random["track_id"])

6pGuOWYOFDcSgmVvFWRWSv 6pGuOWYOFDcSgmVvFWRWSv 1aj1Qg7QvZJ7GZuSY9ISea
6N6BTxTwykM2YI06SeL1ap 6N6BTxTwykM2YI06SeL1ap 5AYu4sjnd68gu3NkfM856Y
2IlL3qHdBURCke78uEdGnM 2IlL3qHdBURCke78uEdGnM 6aXwozVYwf56zRSI5Z3lPK
744ZuzjXQmoJmOdk2I1ym9 744ZuzjXQmoJmOdk2I1ym9 210f6k9YxkeQt3ZThb2wxV
1d6KS9GH06JAd19uiBy9IE 1d6KS9GH06JAd19uiBy9IE 2b6pyyrxfP79LhV2hLtTm4
0fD6vPYWty2Jy4VVozWzfp 0fD6vPYWty2Jy4VVozWzfp 6lxcWIvMQK3yezxwFfZcKZ
741OE401HO7ZCaPhEqCZ7w 741OE401HO7ZCaPhEqCZ7w 1ndyl3wJCFs872XZ3ztPk6
0fD6vPYWty2Jy4VVozWzfp 0fD6vPYWty2Jy4VVozWzfp 7LNI08YZRk67lg7KaPABfg
4U0wNvX09APPfe8TGIYHoO 4U0wNvX09APPfe8TGIYHoO 55IOt0T1KbTgWpsliVyaWo
5MrauZ2u7TRdyWcpdZYhI1 5MrauZ2u7TRdyWcpdZYhI1 4YqCBC4FwzGXuhixt5cgmm
7FxzgizJRGTQ3fxUqfvljg 7FxzgizJRGTQ3fxUqfvljg 7fODjB7BrQTGqh0hogW6XD
1rME1sbMT1519MjbLJrwc7 1rME1sbMT1519MjbLJrwc7 0ZDcLapel9chZ15ZNuOdD4
6dFSbxLsx3x13auTriZLFi 6dFSbxLsx3x13auTriZLFi 4ggiDYAkJ7Yk8d5xOr7Xjo
5lN1EH25gdiqT1SFALMAq1 5lN1EH25gdiqT1SFALMAq1 4wUcvUiC3ljCSJrhwGz6M6
57sk9X1fPLXRfkw74XNrmK 57sk9X1fPLX